# Masked PPO end-to-end training

Train the action-masked PPO policy through the same public API as `aresim-rl train`:

1. resolve the local AresSim package;
2. load and inspect a checked-in experiment YAML;
3. run Ray/RLlib training (optional W&B logging);
4. inspect the run directory and manifest;
5. load the frozen checkpoint as an `Agent`;
6. run framework-neutral validation evaluation.

**Setup** (repository root):

```bash
python3 -m venv engine/.venv
engine/.venv/bin/python -m pip install -e './engine[dev,rllib,notebook]'
engine/.venv/bin/python -m ipykernel install --user --name aresim --display-name "AresSim (.venv)"
```

Select the `AresSim (.venv)` kernel. For `tracking.mode: online`, authenticate once with `engine/.venv/bin/wandb login`.

See [RL Usage Guide](../docs/rl/usage.md) for CLI equivalents and W&B field reference.

In [13]:
from __future__ import annotations

import json
import sys
from dataclasses import replace
from pathlib import Path


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "engine" / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("run this notebook from inside an AresSim checkout")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
ENGINE_DIRECTORY = REPOSITORY_ROOT / "engine"
if str(ENGINE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(ENGINE_DIRECTORY))

try:
    from aresim.training import (
        evaluate_checkpoint,
        load_experiment,
        make_checkpoint_agent,
        run_experiment,
    )
    from aresim.training.experiments import TrackingConfig, apply_overrides
except ModuleNotFoundError as error:
    raise RuntimeError(
        "This notebook requires the rllib extra. From the repository root run:\n"
        "engine/.venv/bin/python -m pip install -e './engine[dev,rllib,notebook]'\n"
        "Then select the 'AresSim (.venv)' kernel."
    ) from error

REPOSITORY_ROOT

PosixPath('/Users/shanmukh/Desktop/Projects/AresSim')

## Parameters

Defaults use the **smoke** config (4,096 training steps, W&B disabled).

**Timing (typical laptop, CPU):**

| Phase | Duration | Controlled by |
|---|---|---|
| Ray startup + PPO training | ~2–4 min | `algorithm_config.total_environment_steps` |
| Post-training validation eval | ~1–3 min with smoke seeds | `RUN_EVALUATION` and `evaluation.*` in YAML |

The smoke config evaluates on **2** validation seeds (not 32) and skips trajectory export. A full dev/reference run with `record_trajectories: true` on the full validation split can take much longer than training.

Change `TRIAL_ID` (or set `artifacts.reject_existing: false` in YAML) before re-running into the same output directory.

In [14]:
CONFIG_PATH = REPOSITORY_ROOT / "configs/masked_ppo/smoke.yaml"
OVERRIDES: tuple[str, ...] = ()  # e.g. ("algorithm_config.learning_rate=0.0001",)

# Post-training frozen eval (2 validation seeds for smoke.yaml). Set False to return right after PPO.
RUN_EVALUATION = False
RUN_REPORT = False

CONFIG_PATH

PosixPath('/Users/shanmukh/Desktop/Projects/AresSim/configs/masked_ppo/smoke.yaml')

## Load and inspect the experiment

The resolved spec is validated before Ray starts. `config_hash` becomes the W&B run id prefix when tracking is enabled.

In [15]:
base_spec = load_experiment(CONFIG_PATH)
spec = apply_overrides(base_spec.as_dict(), OVERRIDES) if OVERRIDES else base_spec
# Always write under <repo>/results even when the notebook cwd is notebooks/.
spec = replace(
    spec,
    artifacts=replace(spec.artifacts, root=str(REPOSITORY_ROOT / "results")),
)

preview = {
    "experiment_id": spec.experiment_id,
    "trial_id": spec.trial_id,
    "algorithm": spec.algorithm,
    "total_environment_steps": spec.algorithm_config.total_environment_steps,
    "evaluation_seed_manifest": spec.evaluation.seed_manifest,
    "record_trajectories": spec.evaluation.record_trajectories,
    "tracking_mode": spec.tracking.mode,
    "tracking_project": spec.tracking.project,
    "artifact_root": spec.artifacts.root,
    "config_hash": spec.config_hash,
    "expected_run_directory": (
        Path(spec.artifacts.root) / spec.experiment_id / spec.trial_id
    ),
}
preview

{'experiment_id': 'rllib_masked_ppo_smoke',
 'trial_id': 'seed_7',
 'algorithm': 'masked_ppo',
 'total_environment_steps': 102400,
 'evaluation_seed_manifest': 'notebooks/phase1_smoke_eval_v1.yaml',
 'record_trajectories': False,
 'tracking_mode': 'disabled',
 'tracking_project': 'aresim',
 'artifact_root': '/Users/shanmukh/Desktop/Projects/AresSim/results',
 'config_hash': 'c0eda16e0ac327d1d5781e460cc1e9809c511488aae814caa6d6a47a844d2c8f',
 'expected_run_directory': PosixPath('/Users/shanmukh/Desktop/Projects/AresSim/results/rllib_masked_ppo_smoke/seed_7')}

## Train

This cell starts Ray, runs RLlib PPO (~2–4 min for smoke), optionally runs frozen validation eval, then shuts Ray down.

If it seems hung after Ray Tune prints `Total run time`, it is usually **not** still training — it is running post-training evaluation (checkpoint rollouts + baseline comparisons). Interrupt the kernel if you only wanted the short PPO smoke.

For online W&B, run `engine/.venv/bin/wandb login` once before executing this cell.

In [16]:
run_directory = run_experiment(
    spec,
    evaluate=RUN_EVALUATION,
    report=RUN_REPORT,
)
run_directory

(PPO pid=83614) 2026-09-02 11:26:13,918	WARNING algorithm_config.py:5160 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
(PPO pid=83614) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
(PPO pid=83614) Install gputil for GPU system monitoring.
2026-09-02 11:29:13,931	WARNING experiment_state.py:209 -- Experiment state snapshotting has been triggered multiple times in the last 5.0 seconds and may become a bottleneck. A snapshot is forced if `CheckpointConfig(num_to_keep)` is set, and a trial has 

PosixPath('/Users/shanmukh/Desktop/Projects/AresSim/results/rllib_masked_ppo_smoke/seed_7')

## Inspect run artifacts

In [17]:
manifest = json.loads((run_directory / "manifest.json").read_text(encoding="utf-8"))
status = json.loads((run_directory / "status.json").read_text(encoding="utf-8"))
checkpoint_sidecar = run_directory / "checkpoints/final/checkpoint.json"

{
    "status": status.get("status"),
    "config_hash": manifest.get("config_hash"),
    "wandb_run_id": manifest.get("wandb_run_id"),
    "checkpoint_exists": checkpoint_sidecar.is_file(),
    "evaluation_dirs": sorted(p.name for p in (run_directory / "evaluation").glob("*") if p.is_dir())
        if (run_directory / "evaluation").exists()
        else [],
}

{'status': 'completed',
 'config_hash': 'c0eda16e0ac327d1d5781e460cc1e9809c511488aae814caa6d6a47a844d2c8f',
 'wandb_run_id': 'c0eda16e0ac327d1d5781e46',
 'checkpoint_exists': True,
 'evaluation_dirs': []}

## Load checkpoint as an Agent

The checkpoint agent implements `aresim.algorithms.Agent` and uses masked argmax by default.

In [18]:
agent = make_checkpoint_agent(checkpoint_sidecar)
{
    "policy_id": agent.policy_id,
    "observation_schema": agent.observation_schema,
    "action_schema": agent.action_schema,
}

{'policy_id': 'rllib_masked_ppo_smoke:seed_7:final',
 'observation_schema': 'aresim.obs.local.v1',
 'action_schema': 'aresim.action.rover.v1'}

## Optional: manual validation evaluation

Skip when `RUN_EVALUATION=True` above — training already ran frozen evaluation. This cell is useful when reloading a completed run.

In [19]:
if not RUN_EVALUATION:
    evaluation_output = evaluate_checkpoint(
        checkpoint_sidecar,
        split="validation",
        output_directory=run_directory / "evaluation" / "manual-validation",
    )
    evaluation_output
else:
    "evaluation already ran during training"

## Enable W&B for the next run

Uncomment and adjust, or switch `CONFIG_PATH` to `configs/masked_ppo/dev.yaml`.

In [20]:
# Example: promote smoke config to an online W&B run with a fresh trial id
# spec = replace(
#     load_experiment(REPOSITORY_ROOT / "configs/masked_ppo/smoke.yaml"),
#     trial_id="notebook-wandb-001",
#     tracking=TrackingConfig(mode="online", project="aresim", tags=("notebook",)),
# )
# run_directory = run_experiment(spec, evaluate=True, report=True)